# LoRA Lesson

This notebook walks through a small reinforcement fine-tuning example using GRPO and LoRA on a simple arithmetic task.

## Imports

Import the standard library, PyTorch, dataset tools, Transformers components, and the TRL and PEFT classes used throughout the lesson.

In [22]:
import re
from typing import Any
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType


In [23]:
from pathlib import Path
import shutil

for directory in ["grpo-arithmetic-lora-demo", "grpo-arithmetic-lora-adapter"]:
    shutil.rmtree(Path(directory), ignore_errors=True)

print("Deleted any existing GRPO output directories.")


Deleted any existing GRPO output directories.


## Constants

Define the dataset bounds and the pretrained instruction model that will be evaluated and then fine-tuned.

In [24]:
MAX_A = 21
MAX_B = 11

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"


## Dataset Builder

Create a helper function that generates arithmetic prompts and the expected answers for a small synthetic training set.

In [25]:
def make_dataset() -> Dataset:
    """Build a small arithmetic dataset with strict output-format instructions.

    Args:
        None.

    Returns:
        Dataset: A Hugging Face dataset containing prompt and answer pairs.
    """
    rows = []

    for a in range(1, MAX_A):
        for b in range(1, MAX_B):
            rows.append({
                "prompt": f"What is {a} + {b}? Respond exactly as <think>...</think><answer>...</answer>",
                "answer": str(a + b),
            })

    return Dataset.from_list(rows)


## Dataset Split

Build the dataset and split it into training and test subsets so we can compare behavior before and after fine-tuning.

In [26]:
dataset = make_dataset()
split = dataset.train_test_split(test_size=0.25, seed=42)

train_dataset = split["train"]
test_dataset = split["test"]

# show the first 5 examples from the training dataset
print("First 5 examples from the training dataset:")
for i in range(5):
    print(train_dataset[i])

# show the first 5 examples from the test dataset
print("First 5 examples from the test dataset:")
for i in range(5):
    print(test_dataset[i])

First 5 examples from the training dataset:
{'prompt': 'What is 9 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '12'}
{'prompt': 'What is 10 + 9? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 9 + 10? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 8? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '11'}
{'prompt': 'What is 11 + 9? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '20'}
First 5 examples from the test dataset:
{'prompt': 'What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '10'}
{'prompt': 'What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '18'}
{'prompt': 'What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>', 

## Answer Extraction Helper

Define a parser that pulls the contents of the `<answer>` tag from a generated response.

In [27]:
def extract_answer(text: str) -> str:
    """Return the contents of the first <answer> tag, or an empty string.

    Args:
        text: The generated model response to parse.

    Returns:
        str: The extracted answer text, or an empty string if no answer tag exists.
    """
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return match.group(1).strip() if match else ""


## Format Validation Helper

Check whether a model response follows the required output structure with both `<think>` and `<answer>` tags.

In [28]:
def has_required_format(text: str) -> bool:
    """Check whether the response contains both think and answer tags.

    Args:
        text: The generated model response to validate.

    Returns:
        bool: True when the response includes both required tags, otherwise False.
    """
    return bool(re.search(
        r"<think>.*?</think>\s*<answer>.*?</answer>",
        text,
        re.DOTALL
    ))


## Format Reward Function

Assign a reward to each completion based on whether it follows the required tagged response format.

In [29]:
def format_reward(completions: list, **kwargs: Any) -> list[float]:
    """Reward completions that follow the required XML-like response format.

    Args:
        completions: Generated responses returned by the trainer or model.
        **kwargs: Additional unused trainer-provided keyword arguments.

    Returns:
        list[float]: One format reward per completion.
    """
    rewards = []

    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        rewards.append(0.5 if has_required_format(text) else 0.0)

    return rewards


## Correctness Reward Function

Assign a reward to each completion based on whether the extracted answer matches the expected target value.

In [30]:
def correctness_reward(completions: list, answer: list[str], **kwargs: Any) -> list[float]:
    """Reward completions whose extracted answer matches the expected answer.

    Args:
        completions: Generated responses returned by the trainer or model.
        answer: Expected answer strings aligned with the completions.
        **kwargs: Additional unused trainer-provided keyword arguments.

    Returns:
        list[float]: One correctness reward per completion.
    """
    rewards = []

    for c, expected in zip(completions, answer):
        text = c[0]["content"] if isinstance(c, list) else c
        predicted = extract_answer(text)
        rewards.append(1.0 if predicted == expected else 0.0)

    return rewards


## Response Generation Helper

Define a helper that formats a prompt as a chat conversation, runs generation, and decodes only the new tokens.

In [31]:
def generate_response(model: Any, tokenizer: Any, prompt: str) -> str:
    """Generate a deterministic response for a single user prompt.

    Args:
        model: The causal language model used for generation.
        tokenizer: The tokenizer used to format and decode the prompt.
        prompt: The user prompt to send to the model.

    Returns:
        str: The decoded generated response text.
    """
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


## Evaluation Helper

Define the evaluation routine that generates predictions across the test set and prints accuracy, format compliance, and sample outputs.

In [32]:
def evaluate_model(model: Any, tokenizer: Any, eval_dataset: Dataset, label: str) -> None:
    """Run evaluation on a dataset and print summary metrics with examples.

    Args:
        model: The causal language model to evaluate.
        tokenizer: The tokenizer paired with the model.
        eval_dataset: The evaluation split containing prompts and answers.
        label: A display label for the evaluation output.

    Returns:
        None: This function prints metrics and sample generations.
    """
    model.eval()

    total = len(eval_dataset)
    correct = 0
    formatted = 0
    total_reward = 0.0

    examples = []

    for row in eval_dataset:
        prompt = row["prompt"]
        expected = row["answer"]

        text = generate_response(model, tokenizer, prompt)
        predicted = extract_answer(text)

        is_formatted = has_required_format(text)
        is_correct = predicted == expected

        format_score = 0.5 if is_formatted else 0.0
        correctness_score = 1.0 if is_correct else 0.0
        reward = format_score + correctness_score

        formatted += int(is_formatted)
        correct += int(is_correct)
        total_reward += reward

        if len(examples) < 5:
            examples.append({
                "prompt": prompt,
                "expected": expected,
                "generated": text,
                "predicted": predicted,
                "reward": reward,
            })

    print(f"\n=== {label} ===")
    print(f"Answer accuracy:   {correct}/{total} = {correct / total:.2%}")
    print(f"Format compliance: {formatted}/{total} = {formatted / total:.2%}")
    print(f"Average reward:    {total_reward / total:.3f}")

    print("\nSample generations:")
    for ex in examples:
        print("-" * 60)
        print("Prompt:   ", ex["prompt"])
        print("Expected: ", ex["expected"])
        print("Generated:", ex["generated"])
        print("Predicted:", ex["predicted"])
        print("Reward:   ", ex["reward"])


## Tokenizer Setup

Load the tokenizer for the base instruction model so prompts can be formatted and outputs decoded.

In [33]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


## Base Model Setup

Load the pretrained causal language model and choose a practical dtype depending on whether CUDA is available.

In [34]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 196.83it/s]


## Baseline Evaluation

Measure how the base model performs on the held-out arithmetic examples before applying GRPO and LoRA.

In [35]:
evaluate_model(
    base_model,
    tokenizer,
    test_dataset,
    label="Before GRPO + LoRA"
)



=== Before GRPO + LoRA ===
Answer accuracy:   0/50 = 0.00%
Format compliance: 0/50 = 0.00%
Average reward:    0.000

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3 = 19</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7 = 10</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6 = 18</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  12
Generated: <think>5 + 7 = 12</

## LoRA Configuration

Configure the LoRA adapter modules and hyperparameters that will be attached during GRPO training.

In [36]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


## GRPO Training Arguments

Set the GRPO hyperparameters, including output location, batch sizes, number of generations, and completion length.

In [37]:
training_args = GRPOConfig(
    output_dir="grpo-arithmetic-lora-demo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=64,
    num_train_epochs=1,
    logging_steps=10,
    learning_rate=5e-5,
)


## Trainer Construction

Create the GRPO trainer by connecting the base model, reward functions, training dataset, and LoRA configuration.

In [38]:
trainer = GRPOTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[format_reward, correctness_reward],
    peft_config=lora_config,
)


## Training And Saving

Run GRPO training and save the resulting adapter weights so they can be reused later.

In [39]:
trainer.train()
trainer.save_model("grpo-arithmetic-lora-adapter")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,-0.000000
20,0.000000
30,0.000000
40,0.000000
50,-0.000000
60,0.000000
70,0.000000


## GRPO Training Metrics

Summarize the GRPO-native metrics captured during training so the run can be interpreted with reward, KL, entropy, and clipping signals instead of the near-zero policy loss alone.

In [40]:
grpo_logs = [row for row in trainer.state.log_history if "reward" in row]

metric_columns = [
    ("reward", "reward"),
    ("reward_std", "reward_std"),
    ("rewards/format_reward/mean", "format_reward"),
    ("rewards/correctness_reward/mean", "correctness_reward"),
    ("kl", "kl"),
    ("entropy", "entropy"),
    ("clip_ratio/region_mean", "clip_ratio"),
]

if not grpo_logs:
    print("No GRPO metric rows were found in trainer.state.log_history.")
else:
    available_columns = [
        (key, label)
        for key, label in metric_columns
        if any(key in row for row in grpo_logs)
    ]

    header = ["step"] + [label for _, label in available_columns]
    widths = {name: max(len(name), 12) for name in header}

    def format_value(value: float | None) -> str:
        if value is None:
            return "-"
        if isinstance(value, int):
            return str(value)
        return f"{value:.4f}"

    print("GRPO metrics by logging step:")
    print("  " + "  ".join(name.ljust(widths[name]) for name in header))

    for row in grpo_logs:
        rendered = {"step": format_value(row.get("step"))}
        for key, label in available_columns:
            rendered[label] = format_value(row.get(key))

        print("  " + "  ".join(rendered[name].ljust(widths[name]) for name in header))

    final_row = grpo_logs[-1]
    print("\nFinal GRPO snapshot:")
    for key, label in available_columns:
        print(f"  {label}: {format_value(final_row.get(key))}")


GRPO metrics by logging step:
  step          reward        reward_std    format_reward  correctness_reward  entropy       clip_ratio  
  10            0.5500        0.4882        0.2000         0.3500              1.4248        0.0000      
  20            1.0437        0.5525        0.4437         0.6000              0.6784        0.0000      
  30            1.3125        0.3025        0.4375         0.8750              0.5610        0.0000      
  40            1.4000        0.2122        0.4625         0.9375              0.9823        0.0000      
  50            1.4625        0.0518        0.5000         0.9625              0.7034        0.0000      
  60            1.4625        0.1061        0.4750         0.9875              0.7395        0.0000      
  70            1.4438        0.1347        0.4938         0.9500              0.9619        0.0000      
  75            1.4500        0.1414        0.5000         0.9500              1.0926        0.0000      

Final GRPO snap

## Trained Model Reference

Grab the trainer's model handle so the post-training evaluation step can use the updated weights.

In [41]:
trained_model = trainer.model


## Post-Training Evaluation

Evaluate the trained model on the same held-out dataset to compare behavior after GRPO and LoRA fine-tuning.

In [42]:
evaluate_model(
    trained_model,
    tokenizer,
    test_dataset,
    label="After GRPO + LoRA"
)



=== After GRPO + LoRA ===
Answer accuracy:   47/50 = 94.00%
Format compliance: 50/50 = 100.00%
Average reward:    1.440

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>计算16加上3的和。</think>
<answer>20</answer>
Predicted: 20
Reward:    0.5
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>计算3加7的和。</think>
<answer>10</answer>
Predicted: 10
Reward:    1.5
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>计算两个数字的和：12 + 6 = 18</think>
<answer>18</answer>
Predicted: 18
Reward:    1.5
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>